# init

In [1]:
# scripts/run_workspace.py
from workspace.workspace import Workspace
from util import create_recipes

# workspace
workspace = Workspace(config_path=["config/base.j2", "config/layout.j2"], port=5000)
# core
core = workspace.components["core"]

❌ core connection failed @ 192.168.254.88
🔵 core simulation api enabled
[Display] socket.io connected
[Display] sending initial snapshot (418 items)
[Display] Running at 60 fps


# parameters

In [2]:
# simulation
simulation = True

# speed factor
speed_factor = 1

# cycle
run_per_rack = 1

# hotel
hotel_levels = 4
hotel_joint_j5 = -40
hotel_joint_j6 = 100

# sbs rack adapter
sbs_adapter_j5 = 38 # 10
sbs_adapter_j0 = -50

# syringe
syringe_padding = 60

# tool rack
tool_rack_joint = [-30, 52.866211, -138.88916, -6.745605, -3.713379, -8.920898, 200]

# tube
tube_list = [f"{r}{c}" for r in "ABCDEF" for c in range(1, 9)]
tube_rack_gravity_offset = 4

# cap
cap_list = [f"{r}{c}" for r in "ABCDEF" for c in range(1, 9)]

# capholder
cap_holder_gravity_offset = -15
cap_holder_tool_tcp_z_offset = 1

# decapper
decapper_tool_tcp_z_offset = -3


# main loop

In [ ]:
# recepies
rcp = create_recipes(workspace, core, speed_factor=speed_factor)

# simulation
if not simulation:
    core.simulation(False)


# pick plate gripper
rcp["tool_rack_2"].pick()

# level, tube_index
level = 0
cap_index = 0
while True:
    # pick from level {i} of hotel
    rcp["hotel"].pick(level)

    # place the sbs plate in
    rcp["sbs_adapter"].place()

    # change the gripper to suction
    rcp["tool_rack_2"].place()
    rcp["tool_rack_1"].pick()

    
    # pick from feeder and place in cap holder
    for index in range(cap_index, cap_index+run_per_rack):
        # above the feeder
        rcp["feeder"].above(anchor="plate_center")
        # detec and present
        rcp["feeder"].present_cap(rcp["inspector"])  
        # pick from feeder
        rcp["feeder"].pick(approach=False)
        # place in cap holder
        rcp["cap_holder"].place(cap_list[index], gravity_offset=cap_holder_gravity_offset)
    
    # place suction gripper
    rcp["tool_rack_1"].place()
    # change the gripper to tube gripper
    rcp["tool_rack_0"].pick()

    # capping
    for index in range(cap_index, cap_index+run_per_rack):
        # pick tube
        rcp["sbs_plate"].pick(tube_list[index])
        # place in decapper
        rcp["decapper"].place()   
        # pick cap
        rcp["cap_holder"].pick(cap_list[index], tool_tcp_z_offset=cap_holder_tool_tcp_z_offset)
        # arm down
        rcp["dispense_arm"].down()
        # dispense
        rcp["dispense_arm"].dispense()
        # arm up
        rcp["dispense_arm"].up()
        # capping
        rcp["decapper"].cap(exit=False)
        # pick from decapper
        rcp["decapper"].pick(approach=False, tool_tcp_z_offset=decapper_tool_tcp_z_offset)
        # back to sbs plate
        rcp["sbs_plate"].place(tube_list[index], gravity_offset=tube_rack_gravity_offset)  
    
    # place the tube gripper
    rcp["tool_rack_0"].place()
    
    # pick plate gripper
    rcp["tool_rack_2"].pick()

    # pick from sbs adaptor
    rcp["sbs_adapter"].pick(anchor="place")

    # place in level {i} of hotel
    rcp["hotel"].place(level)

    # level
    level = (level+1)%hotel_levels
    
    # update cap_index
    if level == 0:
        cap_index += run_per_rack
    
    # exit condition
    if cap_index >= len(cap_list):
        break

# place the plate gripper
rcp["tool_rack_2"].place()